# Explore here

In [ ]:
pip install requests pandas matplotlib seaborn sqlalchemy


Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.1.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import requests

In [ ]:
url= "https://api.worldbank.org/v2/country/chn;ago/indicator/SP.POP.TOTL?date=2010:2024&format=json"
response= requests.get(url)
response.text

'[{"page":1,"pages":1,"per_page":50,"total":30,"sourceid":"2","lastupdated":"2026-04-08"},[{"indicator":{"id":"SP.POP.TOTL","value":"Population, total"},"country":{"id":"AO","value":"Angola"},"countryiso3code":"AGO","date":"2024","value":37885849,"unit":"","obs_status":"","decimal":0},{"indicator":{"id":"SP.POP.TOTL","value":"Population, total"},"country":{"id":"AO","value":"Angola"},"countryiso3code":"AGO","date":"2023","value":36749906,"unit":"","obs_status":"","decimal":0},{"indicator":{"id":"SP.POP.TOTL","value":"Population, total"},"country":{"id":"AO","value":"Angola"},"countryiso3code":"AGO","date":"2022","value":35635029,"unit":"","obs_status":"","decimal":0},{"indicator":{"id":"SP.POP.TOTL","value":"Population, total"},"country":{"id":"AO","value":"Angola"},"countryiso3code":"AGO","date":"2021","value":34532429,"unit":"","obs_status":"","decimal":0},{"indicator":{"id":"SP.POP.TOTL","value":"Population, total"},"country":{"id":"AO","value":"Angola"},"countryiso3code":"AGO","dat

In [ ]:
paises= ['CHL', 'KEN','USA', 'ARG', 'BRA']
indicadores= {'SP.POP.TOTL':'poblacion_total', # Acá están los indicadores en un diccionario.
              'EN.GHG.CO2.PC.CE.AR5':'co2_per_capita',
              'SP.DYN.CBRT.IN':'natalidad_por_mil_habitantes'}

#Función que me traiga los datos de un indicador:
def feth_data_indicador(codigos_pais, id_indicador,fecha_inicio=2010,fecha_fin=2024):
    paises= ";".join(codigos_pais) #Esto me da una lista con punto y coma
    endpoint= f'https://api.worldbank.org/v2/country/{paises}/indicator/{id_indicador}'
    
    pagina= 1
    datos_extendidos= []
    while True:
        params= {
            'format':'json',
            'date': f'{fecha_inicio}:{fecha_fin}',
            'page': pagina,
        }
        response= requests.get(endpoint, params=params)
        data= response.json()

        if not isinstance(data, list) or len(data)==0: # con este if yo estoy controlando dos cosas: que data sea una lista, y que sea más larga que 0. Con esto levanto errores, si no es una lista o si es = a 0, hacemos lo que se llama "levantar un error", con "raise".
            raise ValueError(f'Respuesta de API no esperada {id_indicador}:{data}') #Así se controlan errores, si falla porque no es una lista, o porque tiene largo cero, ya tengo mi error.

        metadatos= data[0]
        datos= data[1] 
        datos_extendidos.extend(datos) 

        total_paginas= metadatos.get('pages') #Se utiliza "get" porque es un diccionario. Con esto tengo el total de páginas

        if pagina >= total_paginas: 
            break #me salgo del while
        pagina +=1 # y si no es así, voy a sumarle 1 a página para que vuelva a entrar al while y agregue a total_paginas todos los datos.

    return datos_extendidos

#Con esto tengo una función para traer los datos de mi indicador. Se definió la función que acabo de crear si corro el código.

In [ ]:


# Ahora vamos a hacer un FOR para recorrer esos indicadores que están en un diccionario. Vamos a obtener los datos de esos indicadores:

tablas= {} #define la tabla como un diccionario vacío
import pandas as pd
for id_indicador, nombre_tabla in indicadores.items: # asi recorro un diccionario
    data_raw = feth_data_indicador(paises, id_indicador) # así tengo mi data cruda o data raw
    
    # y ahora para volverlos un DataFrame debo guardarlos en una lista o diccionario.
    records=[]
    for fila in data_raw:
        records.append({
            'pais': fila['country']['value'],
            'agno': fila['date'],
            'valor': fila['value']
        })
    df= pd.DataFrame(records) #y ahora hacemos que esa lista de diccionarios se vuelva un DataFrame con Pandas

    df['agno'] = pd.to_numeric(df['agno'], errors='coerce').astype('Int64') #Con esto combierto el 'agno' de string a numérico.
    tablas[nombre_tabla] = df
print(df.info())

TypeError: 'builtin_function_or_method' object is not iterable